## 3. Additional topics: neighborhood enrichment & spatially variable genes

This notebook continues from `02_Xenium_5k_downstream_analysis_python.ipynb`: it reuses the single lung sample processed there to demonstrate a simple neighborhood enrichment and spatial autocorrelation analysis.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import spatialdata as sd
import spatialdata_plot
import squidpy as sq

%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore")

In [ ]:
# Read the processed sample saved at the end of Section 2
sdata = sd.read_zarr("../../data/xenium_prime_5k_processed.zarr")
adata = sdata["table"]

### 3.1 Centrality scores

Centrality measures how "important" a cluster is within the spatial neighborhood graph.

- **Average clustering** — tendency of cluster members to form triangles; high values suggest tight-knit, localized tissue regions.
- **Closeness centrality** — how close a cluster is, on average, to all other clusters; a high value may indicate a mediator role in tissue structure.
- **Degree centrality** — fraction of edges from cluster members that connect to other clusters; high values suggest outward-facing influence.

In [ ]:
# Build the spatial neighbor graph (Delaunay triangulation), then compute per-cluster centrality metrics
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)
sq.gr.centrality_scores(adata, cluster_key="leiden")
sq.pl.centrality_scores(adata, cluster_key="leiden", figsize=(16, 5))

### 3.2 Neighborhood enrichment

Neighborhood enrichment scores the proximity of cell clusters on the connectivity graph against a permutation-based null distribution, producing a z-score per cluster pair.

In [ ]:
# Test whether cluster pairs are spatially enriched (permutation-based z-scores)
sq.gr.nhood_enrichment(adata, cluster_key="leiden")

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
sq.pl.nhood_enrichment(
    adata,
    cluster_key="leiden",
    figsize=(8, 8),
    title="Neighborhood enrichment adata",
    ax=ax[0],
)
sdata.pl.render_shapes("cell_boundaries", color="leiden").pl.show(ax=ax[1])

### 3.3 Identify spatially variable genes (Moran's I)

A high Moran's I score indicates strong, non-random, spatially clustered gene expression — a useful signal for tissue organization and spatial domains.

In [ ]:
%%time
# Compute Moran's I spatial autocorrelation per gene to find spatially variable genes
sq.gr.spatial_autocorr(adata, mode="moran", n_jobs=-1)
adata.uns["moranI"].head(10)

In [ ]:
# Auto-pick the top 2 spatially variable genes instead of hardcoding gene names
top_genes = adata.uns["moranI"].index[:2].tolist()

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    color=top_genes,
    shape=None,
    size=2,
    img=False,
    layer="lognorm",
)